In [1]:
import pandas as pd

In [2]:
tips_matrix = pd.read_csv("../02_tips_matrix.csv", index_col=0)
player_lookup = pd.read_csv("../02_player_lookup.csv", index_col=0)["safe_name"].to_dict()
team_lookup = pd.read_csv("00_lookup.csv", index_col=1)["fifa_code"].to_dict()

In [3]:
# teams that didn't make it out of the groups omitted
team_overrides = {
    "Bosnia_and_Herz": "Bosnia and Herzegovina",
    "Cape_Verde": "Cape Verde",
    "DR_Congo": "DR Congo",
    "Ivory_Coast": "Ivory Coast",
    "South_Africa": "South Africa",
    "USA": "United States",
}
for bad, good in team_overrides.items():
    team_lookup[bad] = team_lookup[good]

In [4]:
bracket_tips = tips_matrix[[c for c in tips_matrix.columns if c.startswith("BRACKET")]]
bracket_tips.columns = bracket_tips.columns.str[8:].map(team_lookup)
bracket_tips.columns.name = "team"
bracket_tips = bracket_tips.loc[:, ~bracket_tips.columns.isna()]
bracket_tips.index = bracket_tips.index.map(player_lookup).rename("player")
tip_score = bracket_tips.stack()

In [5]:
stages = pd.DataFrame(index=tip_score.index, columns=pd.Index([], name="stage"))
stages["R32"] = tip_score >= 1
stages["R16"] = tip_score >= 2
stages["quarter"] = tip_score >= 3
stages["semi"] = tip_score >= 4
stages["third"] = (tip_score > 4) & (tip_score < 5)
stages["final"] = tip_score >= 5
stages["winner"] = tip_score == 6

In [6]:
out = stages.stack().loc[lambda x: x].index.to_frame().reset_index(drop=True)
out.to_csv("02_tips.csv", index=False)